# NB3 · Estados financieros mensuales

## Objetivo

Construir las tablas financieras que alimentarán los ratios y Power BI:

- Balance mensual con saldos de cierre;
- PyG mensual y acumulada YTD;
- principales magnitudes financieras;
- PyG por centro de coste;
- control mensual del cuadre contable.

> El Balance se construye con saldos. La PyG se construye con movimientos. Nunca se suman saldos de distintos meses.

In [1]:
# ==========================================================
# 1. LIBRERÍAS Y RUTAS
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


pd.set_option("display.max_columns", 100)
pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ==========================================================
# RUTAS DEL PROYECTO
# ==========================================================

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
elif (CWD.parent / "data").exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Debe existir una carpeta 'data/' en la raíz."
    )


PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = PROCESSED_DIR / "03"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RUTAS = {
    "balance": (
        PROCESSED_DIR
        / "01"
        / "01_balance_saldos_cierre.csv"
    ),

    "movimientos_balance": (
        PROCESSED_DIR
        / "01"
        / "01_balance_movimientos.csv"
    ),

    "maestro": (
        PROCESSED_DIR
        / "02"
        / "02_maestro_financiero.csv"
    ),

    "contabilidad": (
        PROCESSED_DIR
        / "02"
        / "02_contabilidad_financiera.csv"
    ),
}


# ==========================================================
# COMPROBACIÓN
# ==========================================================

for nombre, ruta in RUTAS.items():

    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró '{nombre}':\n{ruta}"
        )


print(f"Raíz del proyecto: {BASE_DIR}")
print(f"Salida NB3:         {OUTPUT_DIR}")

print("\nArchivos encontrados:")

for nombre, ruta in RUTAS.items():
    print(f" - {nombre}: {ruta.name}")

Raíz del proyecto: C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios
Salida NB3:         C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios\data\processed\03

Archivos encontrados:
 - balance: 01_balance_saldos_cierre.csv
 - movimientos_balance: 01_balance_movimientos.csv
 - maestro: 02_maestro_financiero.csv
 - contabilidad: 02_contabilidad_financiera.csv


In [2]:
# ==========================================================
# 2. CARGA Y VALIDACIÓN DE DATOS
# ==========================================================

# ==========================================================
# COLUMNAS NECESARIAS
# ==========================================================

COLUMNAS_BALANCE = [
    "periodo",
    "cuenta_n5",
    "descripcion_n5",
    "saldo_cierre",
]

COLUMNAS_MOVIMIENTOS_BALANCE = [
    "periodo",
    "cuenta_n5",
    "descripcion_n5",
    "movimiento_mes",
]

COLUMNAS_MAESTRO = [
    "cuenta_n5",
    "descripcion_n5",
    "tipo_estado",
    "masa_financiera",
    "categoria_financiera",
    "subcategoria_financiera",
    "naturaleza",
    "circulante",
    "driver_negocio",
    "estado_clasificacion",
]

COLUMNAS_CONTABILIDAD = [
    "periodo",
    "cuenta_n5",
    "centro_coste",
    "tipo_estado",
    "masa_financiera",
    "categoria_financiera",
    "subcategoria_financiera",
    "driver_negocio",
    "estado_clasificacion",
    "importe",
    "importe_analitico",
]


# ==========================================================
# FUNCIÓN DE VALIDACIÓN
# ==========================================================

def validar_columnas_csv(ruta, columnas_esperadas):
    """
    Comprueba que un CSV contiene las columnas mínimas
    necesarias antes de cargarlo.
    """

    disponibles = pd.read_csv(
        ruta,
        nrows=0
    ).columns.tolist()

    faltantes = [
        columna
        for columna in columnas_esperadas
        if columna not in disponibles
    ]

    if faltantes:
        raise ValueError(
            f"El archivo '{ruta.name}' no contiene "
            f"las columnas necesarias: {faltantes}"
        )


# ==========================================================
# VALIDACIÓN PREVIA
# ==========================================================

validar_columnas_csv(
    RUTAS["balance"],
    COLUMNAS_BALANCE
)

validar_columnas_csv(
    RUTAS["movimientos_balance"],
    COLUMNAS_MOVIMIENTOS_BALANCE
)

validar_columnas_csv(
    RUTAS["maestro"],
    COLUMNAS_MAESTRO
)

validar_columnas_csv(
    RUTAS["contabilidad"],
    COLUMNAS_CONTABILIDAD
)


# ==========================================================
# CARGA DE SALDOS DE BALANCE
# ==========================================================

balance = pd.read_csv(
    RUTAS["balance"],
    usecols=COLUMNAS_BALANCE,
    dtype={
        "cuenta_n5": "string",
    },
    parse_dates=[
        "periodo"
    ]
)


# ==========================================================
# CARGA DE MOVIMIENTOS DE BALANCE
# ==========================================================

balance_movimientos = pd.read_csv(
    RUTAS["movimientos_balance"],
    usecols=COLUMNAS_MOVIMIENTOS_BALANCE,
    dtype={
        "cuenta_n5": "string",
    },
    parse_dates=[
        "periodo"
    ]
)


# ==========================================================
# CARGA DEL MAESTRO
# ==========================================================

maestro = pd.read_csv(
    RUTAS["maestro"],
    usecols=COLUMNAS_MAESTRO,
    dtype={
        "cuenta_n5": "string",
    }
)


# ==========================================================
# CARGA DE CONTABILIDAD
# ==========================================================

contabilidad = pd.read_csv(
    RUTAS["contabilidad"],
    usecols=COLUMNAS_CONTABILIDAD,
    dtype={
        "cuenta_n5": "string",
        "centro_coste": "string",
    },
    parse_dates=[
        "periodo"
    ]
)


# ==========================================================
# CONTROLES BÁSICOS
# ==========================================================

if balance.empty:
    raise ValueError(
        "El archivo de saldos de Balance está vacío."
    )

if balance_movimientos.empty:
    raise ValueError(
        "El archivo de movimientos de Balance está vacío."
    )

if maestro.empty:
    raise ValueError(
        "El maestro financiero está vacío."
    )

if contabilidad.empty:
    raise ValueError(
        "La contabilidad financiera está vacía."
    )


# ==========================================================
# CONTROL DE CLASIFICACIÓN
# ==========================================================

movimientos_sin_clasificar = (
    contabilidad[
        "estado_clasificacion"
    ]
    .eq("SIN CLASIFICAR")
    .sum()
)


if movimientos_sin_clasificar > 0:
    raise ValueError(
        f"Existen {movimientos_sin_clasificar:,} movimientos "
        "sin clasificar en la contabilidad."
    )


# ==========================================================
# CONTROL DE FECHAS
# ==========================================================

for nombre, df in {
    "balance": balance,
    "movimientos_balance": balance_movimientos,
    "contabilidad": contabilidad,
}.items():

    if df["periodo"].isna().any():
        raise ValueError(
            f"Existen periodos inválidos en '{nombre}'."
        )


# ==========================================================
# RESUMEN
# ==========================================================

print(
    f"Saldos Balance:      {len(balance):,} filas"
)

print(
    f"Movimientos Balance: {len(balance_movimientos):,} filas"
)

print(
    f"Maestro:             {len(maestro):,} cuentas"
)

print(
    f"Contabilidad:        {len(contabilidad):,} movimientos"
)

print(
    f"Sin clasificar:      {movimientos_sin_clasificar:,}"
)

print(
    "Carga y estructura de entrada validadas correctamente."
)

Saldos Balance:      56,340 filas
Movimientos Balance: 56,340 filas
Maestro:             1,565 cuentas
Contabilidad:        78,688 movimientos
Sin clasificar:      0
Carga y estructura de entrada validadas correctamente.


In [3]:
# ==========================================================
# 3. BALANCE MENSUAL DETALLADO
# ==========================================================

# ==========================================================
# NORMALIZACIÓN
# ==========================================================

balance["cuenta_n5"] = (
    balance["cuenta_n5"]
    .astype("string")
    .str.strip()
    .str.zfill(6)
)

maestro["cuenta_n5"] = (
    maestro["cuenta_n5"]
    .astype("string")
    .str.strip()
    .str.zfill(6)
)


# ==========================================================
# CONTROL DE DUPLICADOS
# ==========================================================

duplicados_balance = balance.duplicated(
    subset=[
        "periodo",
        "cuenta_n5",
    ]
)

if duplicados_balance.any():
    raise ValueError(
        f"El balance contiene "
        f"{duplicados_balance.sum():,} duplicados "
        "por periodo + cuenta_n5."
    )


if maestro["cuenta_n5"].duplicated().any():
    raise ValueError(
        "El maestro financiero contiene "
        "cuentas N5 duplicadas."
    )


# ==========================================================
# INTEGRACIÓN CON EL MAESTRO
# ==========================================================

balance_financiero = balance.merge(
    maestro,
    on="cuenta_n5",
    how="left",
    suffixes=("", "_maestro"),
    validate="many_to_one"
)


# ==========================================================
# CONTROL DE CUENTAS SIN CLASIFICACIÓN
# ==========================================================

sin_clasificacion_balance = (
    balance_financiero[
        "tipo_estado"
    ].isna()
)

if sin_clasificacion_balance.any():

    cuentas_error = (
        balance_financiero.loc[
            sin_clasificacion_balance,
            "cuenta_n5"
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "Existen cuentas de Balance sin clasificación "
        f"financiera: {cuentas_error[:10]}"
    )


# ==========================================================
# DESCRIPCIÓN
# ==========================================================

if "descripcion_n5_maestro" in balance_financiero.columns:

    balance_financiero["descripcion_n5"] = (
        balance_financiero[
            "descripcion_n5"
        ]
        .fillna(
            balance_financiero[
                "descripcion_n5_maestro"
            ]
        )
    )

    balance_financiero = (
        balance_financiero
        .drop(
            columns=[
                "descripcion_n5_maestro"
            ]
        )
    )


# ==========================================================
# SOLO CUENTAS DE BALANCE
# ==========================================================

balance_financiero = (
    balance_financiero.loc[
        balance_financiero[
            "tipo_estado"
        ].eq("Balance")
    ]
    .copy()
)


# ==========================================================
# SIGNO ANALÍTICO
# ==========================================================

es_activo = (
    balance_financiero[
        "masa_financiera"
    ]
    .str.startswith(
        "Activo",
        na=False
    )
)

balance_financiero["importe_balance"] = (
    np.where(
        es_activo,
        balance_financiero["saldo_cierre"],
        -balance_financiero["saldo_cierre"]
    )
)


# ==========================================================
# VARIABLES TEMPORALES
# ==========================================================

balance_financiero["año"] = (
    balance_financiero[
        "periodo"
    ]
    .dt.year
    .astype("int16")
)

balance_financiero["mes_numero"] = (
    balance_financiero[
        "periodo"
    ]
    .dt.month
    .astype("int8")
)

balance_financiero["año_mes"] = (
    balance_financiero[
        "periodo"
    ]
    .dt.strftime("%Y-%m")
)


# ==========================================================
# CONTROL DE CONTINUIDAD
# ==========================================================

periodos_balance = pd.DatetimeIndex(
    balance_financiero[
        "periodo"
    ]
    .drop_duplicates()
    .sort_values()
)

periodos_esperados = pd.date_range(
    start=periodos_balance.min(),
    end=periodos_balance.max(),
    freq="MS"
)

periodos_faltantes_balance = (
    periodos_esperados
    .difference(periodos_balance)
)

if len(periodos_faltantes_balance) > 0:
    raise ValueError(
        "Faltan meses en los saldos de Balance: "
        f"{[
            p.strftime('%Y-%m')
            for p in periodos_faltantes_balance
        ]}"
    )


print(
    f"Meses disponibles: "
    f"{balance_financiero['periodo'].nunique()}"
)

print(
    f"Cuentas de balance: "
    f"{balance_financiero['cuenta_n5'].nunique()}"
)

display(balance_financiero.head())

Meses disponibles: 36
Cuentas de balance: 909


,periodo,cuenta_n5,descripcion_n5,saldo_cierre,tipo_estado,masa_financiera,categoria_financiera,subcategoria_financiera,naturaleza,circulante,driver_negocio,estado_clasificacion,importe_balance,año,mes_numero,año_mes
0,2021-01-01,100000,Capital Social,"-3,000.00",Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA,"3,000.00",2021,1,2021-01
1,2021-01-01,100001,Capital Social por ampliaciones de Capital,"-51,000.00",Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA,"51,000.00",2021,1,2021-01
2,2021-01-01,103000,Socios por desembolsos no exigidos,0.00,Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA,-0.00,2021,1,2021-01
3,2021-01-01,110000,Prima Emisión,"-4,949,000.00",Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA,"4,949,000.00",2021,1,2021-01
4,2021-01-01,112000,Reserva legal,0.00,Balance,Patrimonio neto,Patrimonio neto,Capital y reservas,Acreedora,No corriente,Patrimonio,CLASIFICADA,-0.00,2021,1,2021-01


In [4]:
# ==========================================================
# 4. PYG MENSUAL Y ACUMULADA YTD
# ==========================================================

# ==========================================================
# NORMALIZACIÓN
# ==========================================================

balance_movimientos["cuenta_n5"] = (
    balance_movimientos[
        "cuenta_n5"
    ]
    .astype("string")
    .str.strip()
    .str.zfill(6)
)


# ==========================================================
# CONTROL DE DUPLICADOS
# ==========================================================

duplicados_movimientos = (
    balance_movimientos
    .duplicated(
        subset=[
            "periodo",
            "cuenta_n5",
        ]
    )
)

if duplicados_movimientos.any():
    raise ValueError(
        f"Hay {duplicados_movimientos.sum():,} duplicados "
        "en movimientos de Balance por periodo + cuenta."
    )


# ==========================================================
# INTEGRACIÓN CON MAESTRO
# ==========================================================

columnas_maestro_pyg = [
    "cuenta_n5",
    "descripcion_n5",
    "tipo_estado",
    "masa_financiera",
    "categoria_financiera",
    "subcategoria_financiera",
    "driver_negocio",
]


movimientos_financieros = (
    balance_movimientos
    .merge(
        maestro[
            columnas_maestro_pyg
        ],
        on="cuenta_n5",
        how="left",
        suffixes=(
            "",
            "_maestro"
        ),
        validate="many_to_one"
    )
)


# ==========================================================
# CONTROL DE CUENTAS SIN MAESTRO
# ==========================================================

sin_maestro_movimientos = (
    movimientos_financieros[
        "tipo_estado"
    ].isna()
)

if sin_maestro_movimientos.any():

    cuentas_error = (
        movimientos_financieros.loc[
            sin_maestro_movimientos,
            "cuenta_n5"
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "Existen movimientos de Balance sin "
        "clasificación financiera: "
        f"{cuentas_error[:10]}"
    )


# ==========================================================
# DESCRIPCIÓN
# ==========================================================

if (
    "descripcion_n5_maestro"
    in movimientos_financieros.columns
):

    movimientos_financieros[
        "descripcion_n5"
    ] = (
        movimientos_financieros[
            "descripcion_n5"
        ]
        .fillna(
            movimientos_financieros[
                "descripcion_n5_maestro"
            ]
        )
    )

    movimientos_financieros = (
        movimientos_financieros
        .drop(
            columns=[
                "descripcion_n5_maestro"
            ]
        )
    )


# ==========================================================
# SOLO PYG
# ==========================================================

pyg_detalle = (
    movimientos_financieros.loc[
        movimientos_financieros[
            "tipo_estado"
        ].eq("PyG")
    ]
    .copy()
)


if pyg_detalle.empty:
    raise ValueError(
        "No se han encontrado movimientos de PyG "
        "en Balances.xlsx."
    )


# ==========================================================
# SIGNO ANALÍTICO
# ==========================================================

# Se invierte el signo para su correcto análisis en el powerbi (en contabilidad vuelcan con sentido contable, lo transformamos a sentido ejecutivo)

pyg_detalle[
    "importe_analitico"
] = (
    -pyg_detalle[
        "movimiento_mes"
    ]
)


# ==========================================================
# IMPUESTO SOBRE BENEFICIOS
# ==========================================================

es_impuesto = (
    pyg_detalle[
        "cuenta_n5"
    ]
    .str.startswith(
        "630",
        na=False
    )
)

pyg_detalle.loc[
    es_impuesto,
    "categoria_financiera"
] = "Impuesto sobre beneficios"

pyg_detalle.loc[
    es_impuesto,
    "subcategoria_financiera"
] = "Impuesto sobre beneficios"

pyg_detalle.loc[
    es_impuesto,
    "driver_negocio"
] = "Impuestos"


# ==========================================================
# PYG MENSUAL
# ==========================================================

columnas_pyg = [
    "periodo",
    "masa_financiera",
    "categoria_financiera",
    "subcategoria_financiera",
    "driver_negocio",
]


pyg_mensual = (
    pyg_detalle
    .groupby(
        columnas_pyg,
        as_index=False,
        observed=True,
        dropna=False
    )[
        "importe_analitico"
    ]
    .sum()
    .rename(
        columns={
            "importe_analitico":
            "importe_mes"
        }
    )
)


# ==========================================================
# VARIABLES TEMPORALES
# ==========================================================

pyg_mensual["año"] = (
    pyg_mensual[
        "periodo"
    ]
    .dt.year
    .astype("int16")
)

pyg_mensual[
    "mes_numero"
] = (
    pyg_mensual[
        "periodo"
    ]
    .dt.month
    .astype("int8")
)

pyg_mensual[
    "año_mes"
] = (
    pyg_mensual[
        "periodo"
    ]
    .dt.strftime("%Y-%m")
)


# ==========================================================
# YTD
# ==========================================================

pyg_mensual = (
    pyg_mensual
    .sort_values(
        [
            "año",
            "masa_financiera",
            "categoria_financiera",
            "subcategoria_financiera",
            "driver_negocio",
            "periodo",
        ]
    )
    .reset_index(drop=True)
)


pyg_mensual[
    "importe_ytd"
] = (
    pyg_mensual
    .groupby(
        [
            "año",
            "masa_financiera",
            "categoria_financiera",
            "subcategoria_financiera",
            "driver_negocio",
        ],
        observed=True,
        dropna=False
    )[
        "importe_mes"
    ]
    .cumsum()
)


# ==========================================================
# CONTROL DE CONTINUIDAD
# ==========================================================

periodos_pyg = pd.DatetimeIndex(
    pyg_detalle[
        "periodo"
    ]
    .drop_duplicates()
    .sort_values()
)

periodos_esperados_pyg = pd.date_range(
    start=periodos_pyg.min(),
    end=periodos_pyg.max(),
    freq="MS"
)

periodos_faltantes_pyg = (
    periodos_esperados_pyg
    .difference(periodos_pyg)
)


if len(periodos_faltantes_pyg) > 0:
    raise ValueError(
        "Faltan meses en la PyG: "
        f"{[
            p.strftime('%Y-%m')
            for p in periodos_faltantes_pyg
        ]}"
    )


print(
    f"Meses PyG: "
    f"{pyg_mensual['periodo'].nunique()}"
)

print(
    f"Cuentas PyG: "
    f"{pyg_detalle['cuenta_n5'].nunique()}"
)

display(pyg_mensual.head())

Meses PyG: 36


Cuentas PyG: 655


,periodo,masa_financiera,categoria_financiera,subcategoria_financiera,driver_negocio,importe_mes,año,mes_numero,año_mes,importe_ytd
0,2021-01-01,Gastos,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-162,050.83",2021,1,2021-01,"-162,050.83"
1,2021-02-01,Gastos,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-160,280.95",2021,2,2021-02,"-322,331.78"
2,2021-03-01,Gastos,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-160,904.62",2021,3,2021-03,"-483,236.40"
3,2021-04-01,Gastos,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-159,911.10",2021,4,2021-04,"-643,147.50"
4,2021-05-01,Gastos,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-156,862.55",2021,5,2021-05,"-800,010.05"


In [5]:
# ==========================================================
# 5. MAGNITUDES PRINCIPALES DE LA PYG
# ==========================================================

# ==========================================================
# AGREGACIÓN POR CATEGORÍA
# ==========================================================

pyg_categoria = (
    pyg_mensual
    .groupby(
        [
            "periodo",
            "categoria_financiera",
        ],
        as_index=False,
        observed=True
    )[
        "importe_mes"
    ]
    .sum()
)

pyg_pivot = (
    pyg_categoria
    .pivot(
        index="periodo",
        columns="categoria_financiera",
        values="importe_mes"
    )
    .fillna(0.0)
    .sort_index()
)


# ==========================================================
# FUNCIÓN AUXILIAR
# ==========================================================

def col(nombre):

    if nombre in pyg_pivot.columns:
        return pyg_pivot[nombre]

    return pd.Series(
        0.0,
        index=pyg_pivot.index
    )


# ==========================================================
# CATEGORÍAS
# ==========================================================

ingresos_explotacion = col(
    "Ingresos de explotación"
)

aprovisionamientos = col(
    "Aprovisionamientos"
)

personal = col(
    "Gastos de personal"
)

gastos_operativos = col(
    "Gastos operativos"
)

amortizaciones = col(
    "Amortizaciones"
)

deterioros = col(
    "Deterioros y provisiones"
)

resultado_financiero = col(
    "Resultado financiero"
)

no_recurrentes = col(
    "Resultados no recurrentes"
)

impuesto_beneficios = col(
    "Impuesto sobre beneficios"
)

otros_ingresos = col(
    "Otros ingresos"
)

otros_gastos = col(
    "Otros gastos"
)


# ==========================================================
# MAGNITUDES
# ==========================================================

magnitudes_pyg = pd.DataFrame(
    index=pyg_pivot.index
)

magnitudes_pyg[
    "ingresos_explotacion"
] = ingresos_explotacion

magnitudes_pyg[
    "aprovisionamientos"
] = aprovisionamientos

magnitudes_pyg[
    "margen_bruto"
] = (
    ingresos_explotacion
    + aprovisionamientos
)

magnitudes_pyg[
    "gastos_personal"
] = personal

magnitudes_pyg[
    "otros_gastos_operativos"
] = gastos_operativos

magnitudes_pyg[
    "ebitda"
] = (
    ingresos_explotacion
    + aprovisionamientos
    + personal
    + gastos_operativos
    + otros_ingresos
    + otros_gastos
)

magnitudes_pyg[
    "amortizaciones"
] = amortizaciones

magnitudes_pyg[
    "deterioros"
] = deterioros

magnitudes_pyg[
    "ebit"
] = (
    magnitudes_pyg["ebitda"]
    + magnitudes_pyg["amortizaciones"]
    + magnitudes_pyg["deterioros"]
)

magnitudes_pyg[
    "resultado_financiero"
] = resultado_financiero

magnitudes_pyg[
    "resultado_no_recurrente"
] = no_recurrentes

magnitudes_pyg[
    "resultado_antes_impuestos"
] = (
    magnitudes_pyg["ebit"]
    + magnitudes_pyg["resultado_financiero"]
    + magnitudes_pyg["resultado_no_recurrente"]
)

magnitudes_pyg[
    "impuesto_beneficios"
] = impuesto_beneficios

magnitudes_pyg[
    "resultado_neto"
] = (
    magnitudes_pyg[
        "resultado_antes_impuestos"
    ]
    + magnitudes_pyg[
        "impuesto_beneficios"
    ]
)


# ==========================================================
# PERIODO COMO COLUMNA
# ==========================================================

magnitudes_pyg = (
    magnitudes_pyg
    .reset_index()
    .sort_values("periodo")
    .reset_index(drop=True)
)


# ==========================================================
# VARIABLES TEMPORALES
# ==========================================================

magnitudes_pyg["año"] = (
    magnitudes_pyg[
        "periodo"
    ]
    .dt.year
    .astype("int16")
)

magnitudes_pyg[
    "mes_numero"
] = (
    magnitudes_pyg[
        "periodo"
    ]
    .dt.month
    .astype("int8")
)

magnitudes_pyg[
    "año_mes"
] = (
    magnitudes_pyg[
        "periodo"
    ]
    .dt.strftime("%Y-%m")
)


# ==========================================================
# YTD
# ==========================================================

columnas_acumulables = [
    "ingresos_explotacion",
    "aprovisionamientos",
    "margen_bruto",
    "gastos_personal",
    "otros_gastos_operativos",
    "ebitda",
    "amortizaciones",
    "deterioros",
    "ebit",
    "resultado_financiero",
    "resultado_no_recurrente",
    "resultado_antes_impuestos",
    "impuesto_beneficios",
    "resultado_neto",
]

for columna in columnas_acumulables:

    magnitudes_pyg[
        f"{columna}_ytd"
    ] = (
        magnitudes_pyg
        .groupby(
            "año",
            observed=True
        )[columna]
        .cumsum()
    )


# ==========================================================
# CONTROL DE IDENTIDADES
# ==========================================================

control_ebit = (
    magnitudes_pyg["ebit"]
    - (
        magnitudes_pyg["ebitda"]
        + magnitudes_pyg["amortizaciones"]
        + magnitudes_pyg["deterioros"]
    )
).abs().max()


control_resultado = (
    magnitudes_pyg[
        "resultado_neto"
    ]
    - (
        magnitudes_pyg[
            "resultado_antes_impuestos"
        ]
        + magnitudes_pyg[
            "impuesto_beneficios"
        ]
    )
).abs().max()


if control_ebit > 0.01:
    raise ValueError(
        "La identidad EBITDA → EBIT "
        "no se cumple correctamente."
    )


if control_resultado > 0.01:
    raise ValueError(
        "La identidad Resultado antes de impuestos "
        "→ Resultado neto no se cumple."
    )


print(
    f"Meses en magnitudes PyG: "
    f"{len(magnitudes_pyg)}"
)

display(magnitudes_pyg.tail())

Meses en magnitudes PyG: 36


,periodo,ingresos_explotacion,aprovisionamientos,margen_bruto,gastos_personal,otros_gastos_operativos,ebitda,amortizaciones,deterioros,ebit,resultado_financiero,resultado_no_recurrente,resultado_antes_impuestos,impuesto_beneficios,resultado_neto,año,mes_numero,año_mes,ingresos_explotacion_ytd,aprovisionamientos_ytd,margen_bruto_ytd,gastos_personal_ytd,otros_gastos_operativos_ytd,ebitda_ytd,amortizaciones_ytd,deterioros_ytd,ebit_ytd,resultado_financiero_ytd,resultado_no_recurrente_ytd,resultado_antes_impuestos_ytd,impuesto_beneficios_ytd,resultado_neto_ytd
31,2023-08-01,"1,085,682.81","-26,341.28","1,059,341.53","-271,522.97","-592,456.17","195,362.39","-150,637.40",0.00,"44,724.99",0.00,0.00,"44,724.99","-11,181.25","33,543.74",2023,8,2023-08,"9,883,827.97","-221,920.64","9,661,907.33","-2,126,499.83","-4,193,986.80","3,341,420.70","-1,206,153.70",0.00,"2,135,267.00","668,106.99","-5,551.59","2,797,822.40","-408,906.70","2,388,915.70"
32,2023-09-01,"1,224,763.29","-29,946.08","1,194,817.21","-271,917.64","-559,067.46","363,832.11","-151,542.27",0.00,"212,289.84","-218,508.70","-10,938.84","-17,157.70","4,289.42","-12,868.28",2023,9,2023-09,"11,108,591.26","-251,866.72","10,856,724.54","-2,398,417.47","-4,753,054.26","3,705,252.81","-1,357,695.97",0.00,"2,347,556.84","449,598.29","-16,490.43","2,780,664.70","-404,617.28","2,376,047.42"
33,2023-10-01,"1,149,896.72","-22,914.71","1,126,982.01","-281,203.93","-568,543.32","277,234.76","-152,169.18",0.00,"125,065.58","-1,750.06","-5,400.60","117,914.92","-29,478.73","88,436.19",2023,10,2023-10,"12,258,487.98","-274,781.43","11,983,706.55","-2,679,621.40","-5,321,597.58","3,982,487.57","-1,509,865.15",0.00,"2,472,622.42","447,848.23","-21,891.03","2,898,579.62","-434,096.01","2,464,483.61"
34,2023-11-01,"1,194,151.60","-24,999.52","1,169,152.08","-273,188.02","-565,018.21","330,945.85","-145,920.10",0.00,"185,025.75",-85.37,-539.69,"184,400.69","-46,100.17","138,300.52",2023,11,2023-11,"13,452,639.58","-299,780.95","13,152,858.63","-2,952,809.42","-5,886,615.79","4,313,433.42","-1,655,785.25",0.00,"2,657,648.17","447,762.86","-22,430.72","3,082,980.31","-480,196.18","2,602,784.13"
35,2023-12-01,"979,435.02","-47,911.85","931,523.17","-256,598.35","-313,553.89","361,370.93","-146,633.73",0.00,"214,737.20","-213,522.12","-4,882.83","-3,667.75","24,150.81","20,483.06",2023,12,2023-12,"14,432,074.60","-347,692.80","14,084,381.80","-3,209,407.77","-6,200,169.68","4,674,804.35","-1,802,418.98",0.00,"2,872,385.37","234,240.74","-27,313.55","3,079,312.56","-456,045.37","2,623,267.19"


In [6]:
# ==========================================================
# 6. CONTROL CUENTA 129 VS RESULTADO YTD
# ==========================================================

TOLERANCIA = 0.01

saldo_cuenta_129 = (
    balance_financiero.loc[
        balance_financiero[
            "cuenta_n5"
        ]
        .str.startswith(
            "129",
            na=False
        )
    ]
    .groupby(
        "periodo",
        as_index=False,
        observed=True
    )[
        "importe_balance"
    ]
    .sum()
    .rename(
        columns={
            "importe_balance":
            "saldo_cuenta_129"
        }
    )
)


resultado_ytd = (
    magnitudes_pyg[[
        "periodo",
        "resultado_neto_ytd",
    ]]
    .copy()
)


control_resultado_129 = (
    resultado_ytd
    .merge(
        saldo_cuenta_129,
        on="periodo",
        how="left",
        validate="one_to_one"
    )
)


control_resultado_129[
    "saldo_cuenta_129"
] = (
    control_resultado_129[
        "saldo_cuenta_129"
    ]
    .fillna(0.0)
)


control_resultado_129[
    "diferencia"
] = (
    control_resultado_129[
        "saldo_cuenta_129"
    ]
    - control_resultado_129[
        "resultado_neto_ytd"
    ]
)


control_resultado_129[
    "estado"
] = np.where(
    control_resultado_129[
        "diferencia"
    ].abs() <= TOLERANCIA,
    "COINCIDE",
    "INFORMATIVO"
)


control_resultado_129[
    "año_mes"
] = (
    control_resultado_129[
        "periodo"
    ]
    .dt.strftime("%Y-%m")
)


print(
    "Control informativo: la cuenta 129 puede no "
    "coincidir con el resultado YTD hasta que se "
    "contabilice el asiento de cierre."
)

display(control_resultado_129)

Control informativo: la cuenta 129 puede no coincidir con el resultado YTD hasta que se contabilice el asiento de cierre.


,periodo,resultado_neto_ytd,saldo_cuenta_129,diferencia,estado,año_mes
0,2021-01-01,"22,134.32","-1,521,708.48","-1,543,842.80",INFORMATIVO,2021-01
1,2021-02-01,"-99,205.85","-1,521,708.48","-1,422,502.63",INFORMATIVO,2021-02
2,2021-03-01,"-229,090.47","-1,521,708.48","-1,292,618.01",INFORMATIVO,2021-03
3,2021-04-01,"-124,914.57","-1,521,708.48","-1,396,793.91",INFORMATIVO,2021-04
4,2021-05-01,"-21,208.55","-1,521,708.48","-1,500,499.93",INFORMATIVO,2021-05
5,2021-06-01,"-335,640.95",0.00,"335,640.95",INFORMATIVO,2021-06
6,2021-07-01,"-200,765.98",0.00,"200,765.98",INFORMATIVO,2021-07
7,2021-08-01,"-144,709.44",0.00,"144,709.44",INFORMATIVO,2021-08
8,2021-09-01,"-422,710.72",0.00,"422,710.72",INFORMATIVO,2021-09
9,2021-10-01,"-298,378.45",0.00,"298,378.45",INFORMATIVO,2021-10


In [7]:
# ==========================================================
# 7. BALANCE MENSUAL Y CONTROL DE CUADRE
# ==========================================================

# ==========================================================
# BALANCE AGRUPADO
# ==========================================================

balance_mensual = (
    balance_financiero
    .groupby(
        [
            "periodo",
            "masa_financiera",
            "categoria_financiera",
            "subcategoria_financiera",
        ],
        as_index=False,
        dropna=False,
        observed=True
    )[
        "importe_balance"
    ]
    .sum()
)


# ==========================================================
# RESULTADO DEL EJERCICIO
# ==========================================================

resultado_balance = (
    magnitudes_pyg[[
        "periodo",
        "resultado_neto_ytd",
    ]]
    .copy()
    .rename(
        columns={
            "resultado_neto_ytd":
            "importe_balance"
        }
    )
)

resultado_balance[
    "masa_financiera"
] = "Patrimonio neto"

resultado_balance[
    "categoria_financiera"
] = "Patrimonio neto"

resultado_balance[
    "subcategoria_financiera"
] = "Resultado del ejercicio"


balance_mensual = pd.concat(
    [
        balance_mensual,
        resultado_balance[
            [
                "periodo",
                "masa_financiera",
                "categoria_financiera",
                "subcategoria_financiera",
                "importe_balance",
            ]
        ],
    ],
    ignore_index=True
)


# ==========================================================
# VARIABLES TEMPORALES
# ==========================================================

balance_mensual["año"] = (
    balance_mensual[
        "periodo"
    ]
    .dt.year
    .astype("int16")
)

balance_mensual[
    "mes_numero"
] = (
    balance_mensual[
        "periodo"
    ]
    .dt.month
    .astype("int8")
)

balance_mensual[
    "año_mes"
] = (
    balance_mensual[
        "periodo"
    ]
    .dt.strftime("%Y-%m")
)


balance_mensual = (
    balance_mensual
    .sort_values(
        [
            "periodo",
            "masa_financiera",
            "categoria_financiera",
            "subcategoria_financiera",
        ]
    )
    .reset_index(drop=True)
)


# ==========================================================
# CONTROL DE CUADRE
# ==========================================================

totales_balance = (
    balance_mensual
    .groupby(
        [
            "periodo",
            "masa_financiera",
        ],
        as_index=False,
        observed=True
    )[
        "importe_balance"
    ]
    .sum()
    .pivot(
        index="periodo",
        columns="masa_financiera",
        values="importe_balance"
    )
    .fillna(0.0)
)


def total_masa(nombre):

    if nombre in totales_balance.columns:
        return totales_balance[nombre]

    return pd.Series(
        0.0,
        index=totales_balance.index
    )

control_cuadre = pd.DataFrame(
    index=totales_balance.index
)

control_cuadre[
    "activo_corriente"
] = total_masa(
    "Activo corriente"
)

control_cuadre[
    "activo_no_corriente"
] = total_masa(
    "Activo no corriente"
)

control_cuadre[
    "pasivo_corriente"
] = total_masa(
    "Pasivo corriente"
)

control_cuadre[
    "pasivo_no_corriente"
] = total_masa(
    "Pasivo no corriente"
)

control_cuadre[
    "patrimonio_neto"
] = total_masa(
    "Patrimonio neto"
)


control_cuadre[
    "activo_total"
] = (
    control_cuadre[
        "activo_corriente"
    ]
    + control_cuadre[
        "activo_no_corriente"
    ]
)


control_cuadre[
    "pasivo_total"
] = (
    control_cuadre[
        "pasivo_corriente"
    ]
    + control_cuadre[
        "pasivo_no_corriente"
    ]
)


control_cuadre[
    "pasivo_pn_total"
] = (
    control_cuadre[
        "pasivo_total"
    ]
    + control_cuadre[
        "patrimonio_neto"
    ]
)


control_cuadre[
    "diferencia"
] = (
    control_cuadre[
        "activo_total"
    ]
    - control_cuadre[
        "pasivo_pn_total"
    ]
)


control_cuadre[
    "estado"
] = np.where(
    control_cuadre[
        "diferencia"
    ].abs() <= TOLERANCIA,
    "CUADRA",
    "REVISAR"
)


# ==========================================================
# PERIODO COMO COLUMNA
# ==========================================================

control_cuadre = (
    control_cuadre
    .reset_index()
    .sort_values("periodo")
    .reset_index(drop=True)
)


control_cuadre["año"] = (
    control_cuadre[
        "periodo"
    ]
    .dt.year
    .astype("int16")
)

control_cuadre[
    "mes_numero"
] = (
    control_cuadre[
        "periodo"
    ]
    .dt.month
    .astype("int8")
)

control_cuadre[
    "año_mes"
] = (
    control_cuadre[
        "periodo"
    ]
    .dt.strftime("%Y-%m")
)


# ==========================================================
# VALIDACIÓN CRÍTICA
# ==========================================================

meses_no_cuadran = (
    control_cuadre[
        "estado"
    ]
    .eq("REVISAR")
    .sum()
)


print("Resultado del control:")

print(
    control_cuadre[
        "estado"
    ]
    .value_counts(
        dropna=False
    )
)


if meses_no_cuadran > 0:

    display(
        control_cuadre.loc[
            control_cuadre[
                "estado"
            ].eq("REVISAR")
        ]
    )

    raise ValueError(
        f"El Balance no cuadra en "
        f"{meses_no_cuadran} meses. "
        "No se continuará con el pipeline."
    )


display(control_cuadre)

Resultado del control:
estado
CUADRA    36
Name: count, dtype: int64


,periodo,activo_corriente,activo_no_corriente,pasivo_corriente,pasivo_no_corriente,patrimonio_neto,activo_total,pasivo_total,pasivo_pn_total,diferencia,estado,año,mes_numero,año_mes
0,2021-01-01,"2,987,136.88","21,322,881.77","1,442,961.27","23,282,366.73","-415,309.35","24,310,018.65","24,725,328.00","24,310,018.65",0.00,CUADRA,2021,1,2021-01
1,2021-02-01,"3,077,633.16","21,204,609.81","1,536,525.76","23,282,366.73","-536,649.52","24,282,242.97","24,818,892.49","24,282,242.97",0.00,CUADRA,2021,2,2021-02
2,2021-03-01,"3,198,649.78","21,098,306.45","1,681,123.64","23,282,366.73","-666,534.14","24,296,956.23","24,963,490.37","24,296,956.23",0.00,CUADRA,2021,3,2021-03
3,2021-04-01,"2,780,444.81","20,962,507.51","1,022,943.83","23,282,366.73","-562,358.24","23,742,952.32","24,305,310.56","23,742,952.32",0.00,CUADRA,2021,4,2021-04
4,2021-05-01,"3,029,571.38","20,806,279.32","1,012,136.19","23,282,366.73","-458,652.22","23,835,850.70","24,294,502.92","23,835,850.70",-0.00,CUADRA,2021,5,2021-05
5,2021-06-01,"3,332,587.37","20,706,300.75","1,529,606.01","23,282,366.73","-773,084.62","24,038,888.12","24,811,972.74","24,038,888.12",0.00,CUADRA,2021,6,2021-06
6,2021-07-01,"4,208,066.91","20,537,032.65","1,336,542.48","23,282,366.73","126,190.35","24,745,099.56","24,618,909.21","24,745,099.56",-0.00,CUADRA,2021,7,2021-07
7,2021-08-01,"3,633,423.49","20,389,168.06","1,482,377.93","22,282,366.73","257,846.89","24,022,591.55","23,764,744.66","24,022,591.55",0.00,CUADRA,2021,8,2021-08
8,2021-09-01,"3,544,103.46","20,237,810.89","1,519,702.01","22,282,366.73","-20,154.39","23,781,914.35","23,802,068.74","23,781,914.35",0.00,CUADRA,2021,9,2021-09
9,2021-10-01,"3,197,779.05","20,115,572.70","926,807.14","22,282,366.73","104,177.88","23,313,351.75","23,209,173.87","23,313,351.75",0.00,CUADRA,2021,10,2021-10


In [8]:
# ==========================================================
# 8. PYG POR CENTRO DE COSTE
# ==========================================================

# Para centro de coste necesitamos la contabilidad,
# porque el archivo de Balance no contiene esta dimensión.

pyg_detalle_cc = contabilidad.loc[
    contabilidad["tipo_estado"].eq("PyG")
].copy()


# ==========================================================
# IMPUESTO SOBRE BENEFICIOS
# ==========================================================

es_impuesto = (
    pyg_detalle_cc["cuenta_n5"]
    .str.startswith(
        "630",
        na=False
    )
)

pyg_detalle_cc.loc[
    es_impuesto,
    "categoria_financiera"
] = "Impuesto sobre beneficios"

pyg_detalle_cc.loc[
    es_impuesto,
    "subcategoria_financiera"
] = "Impuesto sobre beneficios"

pyg_detalle_cc.loc[
    es_impuesto,
    "driver_negocio"
] = "Impuestos"

del es_impuesto


# ==========================================================
# AGREGACIÓN POR CENTRO
# ==========================================================

pyg_centro_coste = (
    pyg_detalle_cc
    .groupby(
        [
            "periodo",
            "centro_coste",
            "categoria_financiera",
            "subcategoria_financiera",
            "driver_negocio",
        ],
        as_index=False,
        observed=True,
        dropna=False
    )["importe_analitico"]
    .sum()
    .rename(
        columns={
            "importe_analitico":
            "importe_mes"
        }
    )
)


# ==========================================================
# VARIABLES TEMPORALES
# ==========================================================

pyg_centro_coste["año"] = (
    pyg_centro_coste["periodo"]
    .dt.year
    .astype("int16")
)

pyg_centro_coste["mes_numero"] = (
    pyg_centro_coste["periodo"]
    .dt.month
    .astype("int8")
)

pyg_centro_coste["año_mes"] = (
    pyg_centro_coste["periodo"]
    .dt.strftime("%Y-%m")
)


# ==========================================================
# YTD
# ==========================================================

pyg_centro_coste = (
    pyg_centro_coste
    .sort_values(
        [
            "año",
            "centro_coste",
            "categoria_financiera",
            "subcategoria_financiera",
            "driver_negocio",
            "periodo",
        ]
    )
    .reset_index(drop=True)
)


pyg_centro_coste["importe_ytd"] = (
    pyg_centro_coste
    .groupby(
        [
            "año",
            "centro_coste",
            "categoria_financiera",
            "subcategoria_financiera",
            "driver_negocio",
        ],
        observed=True,
        dropna=False
    )["importe_mes"]
    .cumsum()
)


display(pyg_centro_coste.head())

,periodo,centro_coste,categoria_financiera,subcategoria_financiera,driver_negocio,importe_mes,año,mes_numero,año_mes,importe_ytd
0,2021-01-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-7,387.34",2021,1,2021-01,"-7,387.34"
1,2021-02-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-7,278.81",2021,2,2021-02,"-14,666.15"
2,2021-03-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-7,041.78",2021,3,2021-03,"-21,707.93"
3,2021-04-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-7,074.80",2021,4,2021-04,"-28,782.73"
4,2021-05-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-6,813.84",2021,5,2021-05,"-35,596.57"


In [9]:
# ==========================================================
# 9. EXPORTACIÓN
# ==========================================================

OUTPUT_DIR = PROCESSED_DIR / "03"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


EXPORTACIONES = {
    "03_balance_detalle_mensual.csv": balance_financiero,
    "03_balance_mensual.csv": balance_mensual,
    "03_control_cuadre_balance.csv": control_cuadre,
    "03_control_resultado_129.csv": control_resultado_129,
    "03_pyg_mensual.csv": pyg_mensual,
    "03_magnitudes_pyg.csv": magnitudes_pyg,
    "03_pyg_centro_coste.csv": pyg_centro_coste,
}


for nombre, df in EXPORTACIONES.items():

    df.to_csv(
        OUTPUT_DIR / nombre,
        index=False,
        encoding="utf-8-sig",
        decimal="."
    )


print("Archivos exportados:")

for nombre in EXPORTACIONES:
    print(f" - data/processed/03/{nombre}")

Archivos exportados:
 - data/processed/03/03_balance_detalle_mensual.csv
 - data/processed/03/03_balance_mensual.csv
 - data/processed/03/03_control_cuadre_balance.csv
 - data/processed/03/03_control_resultado_129.csv
 - data/processed/03/03_pyg_mensual.csv
 - data/processed/03/03_magnitudes_pyg.csv
 - data/processed/03/03_pyg_centro_coste.csv


## Salidas principales

- `03_balance_detalle_mensual.csv`: saldo por cuenta y mes.
- `03_balance_mensual.csv`: Balance agrupado por masas y categorías.
- `03_control_cuadre_balance.csv`: control `Activo = Pasivo + Patrimonio neto`.
- `03_pyg_mensual.csv`: PyG mensual y YTD por categoría.
- `03_magnitudes_pyg.csv`: ingresos, margen bruto, EBITDA, EBIT y resultado.
- `03_pyg_centro_coste.csv`: PyG mensual y YTD por centro de coste.

### Criterios utilizados

- Los saldos del Balance no se acumulan entre meses.
- Los movimientos de PyG sí se acumulan para obtener el YTD.
- El resultado del ejercicio se calcula desde la PyG y se incorpora al patrimonio neto.
- El Balance se analiza a nivel empresa.
- Los centros de coste se utilizan únicamente para ingresos y gastos.